# Projet *Steam's video games platform* – Certification CDSD – bloc 2

<img src="https://full-stack-assets.s3.eu-west-3.amazonaws.com/images/Steam_2016_logo_black.png" alt="Steam" width="295" height="90">

Auteur : Yoann ROBERT

Date de la présentation : 16 juin 2026

## Introduction

### Contexte

Ce projet est réalisé pour le compte d'Ubisoft, un éditeur de jeux vidéo français qui souhaite sortir un nouveau jeu vidéo révolutionnaire. Pour préparer ce lancement, l'entreprise souhaite mieux comprendre l'écosystème du jeu vidéo et les tendances actuelles du marché.

Pour répondre à ce besoin, nous nous appuyons sur les données de Steam, service de distribution numérique et boutique de jeux vidéo édité par Valve. Lancé en septembre 2003, Steam distribue aujourd'hui des milliers de titres et constitue une source de données particulièrement riche pour analyser le marché du jeu vidéo à grande échelle.

### Dataset

Le [dataset](https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json) recense les jeux disponibles sur la marketplace de Steam. Chaque enregistrement décrit un jeu et ses caractéristiques : éditeur, prix, genres, plateformes supportées, langues, notes des utilisateurs, date de sortie, restrictions d'âge, etc.

Il s'agit d'un fichier semi-structuré au format JSON, doté d'un schéma imbriqué (structures et tableaux sur plusieurs niveaux). Il contient également des champs de type texte et date qui nécessitent un traitement spécifique. La manipulation de ces données mobilise notamment les méthodes PySpark `getField()` et `explode()`, ainsi que les fonctions utilitaires de manipulation de texte et de dates.

### Objectifs

L'objectif de ce projet est de comprendre quels facteurs influencent la popularité ou les ventes d'un jeu vidéo. Au-delà de cette question, il s'agit également de dresser une analyse globale du marché du jeu vidéo afin d'éclairer la stratégie d'Ubisoft. Pour cela, l'étude adopte différents niveaux d'analyse (macroscopique, par genres, par plateformes) en s'appuyant sur des agrégations et des analyses segmentées.

### Indications

Pour guider l'étude, plusieurs questions sont proposées selon différents niveaux d'analyse. À titre d'exemple :
- Quel éditeur a publié le plus de jeux sur Steam ?
- Quels sont les genres les plus représentés et les plus lucratifs ?
- Les jeux sont-ils majoritairement disponibles sur Windows, Mac ou Linux ?

### Scope

Cette analyse exploratoire (EDA) est réalisée avec Databricks et PySpark. Les visualisations sont produites à l'aide de l'outil de *dashboarding* de Databricks.

### Livrable

Un ou plusieurs notebooks intégrant la manipulation des données avec PySpark et les visualisations avec l'outil de dashboarding de Databricks, publiés via le bouton "publish" de Databricks afin que le jury puisse accéder à l'ensemble des visualisations.

### Plan de l'étude

L'étude se déroule selon le plan suivant :
- 0. Nettoyage des données
- 1. Analyse macroscopique
- 2. Analyse des genres
- 3. Analyse des plateformes
- 4. Pour aller plus loin
- 5. Conclusion et recommandations à Ubisoft

---

## Configuration

### Imports des libraries

In [0]:
import pyspark
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, DoubleType, LongType, DecimalType

### Note sur la mise en cache

Ce notebook est exécuté sur **Databricks Free Edition**, qui ne donne accès qu'à du *serverless compute*. Sur ce mode de calcul, les API de mise en cache de DataFrame (`cache()`, `persist()`, `unpersist()`) sont [explicitement non supportées](https://docs.databricks.com/aws/en/compute/serverless/limitations) et lèvent une exception.

L'environnement serverless gère son propre cache de manière transparente, sans intervention possible de l'utilisateur. Sur un cluster classique, le DataFrame nettoyé `df` aurait pu être mis en cache après la section 0 afin d'éviter le re-calcul du pipeline de nettoyage à chaque action downstream. Il est réutilisé par une quinzaine d'analyses dans la suite du notebook.

L'alternative officiellement recommandée par Databricks pour ce cas, en environnement serverless, est la **matérialisation** des DataFrames intermédiaires dans une table Delta (`df.write.saveAsTable(...)` puis `spark.table(...)`). Cette option n'a pas été retenue ici car le volume traité (environ 55 000 lignes après nettoyage) ne justifie pas la complexité ajoutée.

### Constantes

In [0]:
INPUT_FILE = "s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/steam_game_output.json"

---

## 0. Nettoyage des données

### 0.1 Chargement des données depuis le bucket AWS S3

In [0]:
df_init = spark.read.format('json').load(INPUT_FILE)

### 0.2 Données caractéristiques du jeu de données

In [0]:
print(f"Nombre d'enregistrements : {df_init.count()}")

Nombre d'enregistrements : 55691


In [0]:
print("Schéma du dataframe :")
df_init.printSchema()

Schéma du dataframe :
root
 |-- data: struct (nullable = true)
 |    |-- appid: long (nullable = true)
 |    |-- categories: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- ccu: long (nullable = true)
 |    |-- developer: string (nullable = true)
 |    |-- discount: string (nullable = true)
 |    |-- genre: string (nullable = true)
 |    |-- header_image: string (nullable = true)
 |    |-- initialprice: string (nullable = true)
 |    |-- languages: string (nullable = true)
 |    |-- name: string (nullable = true)
 |    |-- negative: long (nullable = true)
 |    |-- owners: string (nullable = true)
 |    |-- platforms: struct (nullable = true)
 |    |    |-- linux: boolean (nullable = true)
 |    |    |-- mac: boolean (nullable = true)
 |    |    |-- windows: boolean (nullable = true)
 |    |-- positive: long (nullable = true)
 |    |-- price: string (nullable = true)
 |    |-- publisher: string (nullable = true)
 |    |-- release_date: string (nul

Premier niveau du dataframe :
- `id` → type `string`, type primitif / scalaire, i.e. un champ qui ne contient qu'une seule valeur.
- `data` → type `struct`, type structuré ou *champ imbriqué*, i.e. un champ qui contient lui-même des sous-champs comme `appid` (type `long`, un entier long), `categories` (type `array` - tableau), `developer` (type `string`, chaîne de caractères), `platform` (type `struct`, donc un champ imbriqué dans un champ imbriqué), etc.

In [0]:
print("Aperçu des cinq premiers enregistrements :")
df_init.select(
    F.col("id"),
    F.col("data").getField("name").alias("name"),
    F.col("data").getField("developer").alias("developer"),
    F.col("data").getField("release_date").alias("release_date"),
    F.col("data").getField("price").alias("price"),
    F.col("data").getField("required_age").alias("required_age"),
    F.col("data").getField("positive").alias("positive"),
    F.col("data").getField("negative").alias("negative")
).show(5, truncate=False)

Aperçu des cinq premiers enregistrements :
+-------+---------------------------+----------------------+------------+-----+------------+--------+--------+
|id     |name                       |developer             |release_date|price|required_age|positive|negative|
+-------+---------------------------+----------------------+------------+-----+------------+--------+--------+
|10     |Counter-Strike             |Valve                 |2000/11/1   |999  |0           |201215  |5199    |
|1000000|ASCENXION                  |IndigoBlue Game Studio|2021/05/14  |999  |0           |27      |5       |
|1000010|Crown Trick                |NEXT Studios          |2020/10/16  |599  |0           |4032    |646     |
|1000030|Cook, Serve, Delicious! 3?!|Vertigo Gaming Inc.   |2020/10/14  |1999 |0           |1575    |115     |
|1000040|细胞战争                   |DoubleC Games         |2019/03/30  |199  |0           |0       |1       |
+-------+---------------------------+----------------------+------------+

### 0.3 Suppression de données inutiles à l'étude

In [0]:
print("Affichage des types d'enregistrements présents :")
df_init \
    .select(F.col("data").getField("type").alias("type")) \
    .groupBy("type") \
    .count() \
    .withColumn(
        "action",
        F.when(
            F.col("type") == "game", "à garder"
        ).otherwise("à supprimer")
    ) \
    .show()

Affichage des types d'enregistrements présents :
+--------+-----+-----------+
|    type|count|     action|
+--------+-----+-----------+
|    game|55690|   à garder|
|hardware|    1|à supprimer|
+--------+-----+-----------+



Un enregistrement n'est pas un jeu mais de type `hardware`. Il ne sera pas conservé.

Nous filtrons les entrées pour ne conserver que le type `game`.

Dans la même commande, nous sélectionnons les colonnes utiles pour l'étude.
Les colonnes non conservées sont `id` et les colonnes imbriquées dans la colonne `data` suivantes :
- `ccu`
- `header_image`
- `short_description`
- `tags`
- `website`

In [0]:
df = df_init \
        .filter(F.col("data").getField("type") == "game") \
        .select(
            F.col("data").getField("appid").alias("appid"),
            F.col("data").getField("categories").alias("categories"),
            F.col("data").getField("developer").alias("developer"),
            F.col("data").getField("discount").alias("discount"),
            F.col("data").getField("genre").alias("genre"),
            F.col("data").getField("initialprice").alias("initialprice"),
            F.col("data").getField("languages").alias("languages"),
            F.col("data").getField("name").alias("name"),
            F.col("data").getField("negative").alias("negative"),
            F.col("data").getField("owners").alias("owners"),
            F.col("data").getField("platforms").getField("windows").alias("windows"),
            F.col("data").getField("platforms").getField("mac").alias("mac"),
            F.col("data").getField("platforms").getField("linux").alias("linux"),
            F.col("data").getField("positive").alias("positive"),
            F.col("data").getField("price").alias("price"),
            F.col("data").getField("publisher").alias("publisher"),
            F.col("data").getField("release_date").alias("release_date"),
            F.col("data").getField("required_age").alias("required_age")
        )

NB_GAMES = df.count()
print(f"Nombre de jeux dans le dataframe : {NB_GAMES}")

Nombre de jeux dans le dataframe : 55690


In [0]:
print("Nouveau schéma du dataframe :")
df.printSchema()

Nouveau schéma du dataframe :
root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- developer: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- languages: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- owners: string (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- price: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- required_age: string (nullable = true)



### 0.4 Transformation des données

#### 0.4.1 Données liées au prix

Il reste des étapes dans le nettoyage.
Tout d'abord, les colonnes associées à un prix, `price`, `initialprice` et `discount` n'ont pas un type numérique car elles sont de type `string`.
Pour les deux premières, nous les transformons en `DecimalType(6, 2)`, qui nous permet de bien définir tous les montants entre 0,00 USD et 9 999,99 USD, ce qui est largement suffisant pour les données observées.
Pour la dernière, nous la transformons en `DecimalType(5, 2)` pour parcourir les pourcentages possibles entre 0 et 100 avec deux décimales.

Par ailleurs, après une rapide analyse, il s'avère que les prix `price` et `initialprice` sont indiqués en centièmes de dollar US (USD). Nous les convertissons en USD et ajoutons le suffixe au nom de colonne pour éviter toute confusion ultérieure. De même, la colonne `discount` est un pourcentage : nous ajoutons le suffixe `_PCT`. Les trois colonnes initiales sont supprimées.

Pour faciliter la transformation `string` → `DecimalType`, une étape intermédiaire est ajoutée où les colonnes sont transformées en `FloatType`.

In [0]:
df = df \
    .withColumn(
        "price_USD",
        (F.round(F.col("price").cast(FloatType()) / 100, 2)).cast(DecimalType(6, 2))
    ) \
    .withColumn(
        "initialprice_USD",
        (F.round(F.col("initialprice").cast(FloatType()) / 100, 2)).cast(DecimalType(6, 2))
    ) \
    .withColumn(
        "discount_PCT",
        (F.round(F.col("discount").cast(FloatType()), 2)).cast(DecimalType(5, 2))
    ) \
    .drop(*["price", "initialprice", "discount"])

df.select("name", "price_USD", "initialprice_USD", "discount_PCT").show(5, truncate=False)

+---------------------------+---------+----------------+------------+
|name                       |price_USD|initialprice_USD|discount_PCT|
+---------------------------+---------+----------------+------------+
|Counter-Strike             |9.99     |9.99            |0.00        |
|ASCENXION                  |9.99     |9.99            |0.00        |
|Crown Trick                |5.99     |19.99           |70.00       |
|Cook, Serve, Delicious! 3?!|19.99    |19.99           |0.00        |
|细胞战争                   |1.99     |1.99            |0.00        |
+---------------------------+---------+----------------+------------+
only showing top 5 rows


#### 0.4.2 Date de parution

La colonne `release_date` est également de type `string` et est donc difficilement exploitable. Il faut la convertir au format `date`.

In [0]:
df = df.withColumn(
    "release_date_normalized",
    F.when(
        (F.col("release_date").isNull()) | (F.trim(F.col("release_date")) == ""),
        None
    ).when(
        F.col("release_date").rlike(r"^\d{4}/\d{1,2}$"),  # Quand le jour est manquant, 
        F.concat(F.col("release_date"), F.lit("/01"))     # le premier jour du mois est choisi.
    ) \
    .otherwise(F.col("release_date"))
)

df = df \
    .withColumn(
        "release_date",
        F.to_date(F.col("release_date_normalized"), "yyyy/M/d")
    ) \
    .drop("release_date_normalized")

dates_stats = df \
    .agg(
        F.sum(F.when(F.col("release_date").isNull(), 1).otherwise(0)).alias("nulls"),
        F.min("release_date").alias("oldest"),
        F.max("release_date").alias("latest")
    ) \
    .collect()[0]

nb_nulls_in_release_date = dates_stats["nulls"]
oldest_date = dates_stats["oldest"]
latest_date = dates_stats["latest"]

print(f"Oldest release date: {oldest_date}")
print(f"Latest release date: {latest_date}")
print(
    "Nombre de NULL dans la colonne `release_date': " +
    f"{nb_nulls_in_release_date} / {NB_GAMES} " +
    f"({nb_nulls_in_release_date / NB_GAMES * 100:.2f}%)"
    )

Oldest release date: 1997-06-30
Latest release date: 2022-11-11
Nombre de NULL dans la colonne `release_date': 99 / 55690 (0.18%)


#### 0.4.3 Âge minimum requis

Affichage des valeurs de l'âge minimal obligatoire des jeux :

In [0]:
df \
    .select("required_age") \
    .distinct() \
    .orderBy("required_age") \
    .sort(F.col("required_age")) \
    .show()

+------------+
|required_age|
+------------+
|           0|
|          10|
|          12|
|          13|
|          14|
|          15|
|          16|
|          17|
|          18|
|         180|
|          20|
|         21+|
|           3|
|          35|
|           5|
|           6|
|           7|
|          7+|
|           8|
|           9|
+------------+
only showing top 20 rows


In [0]:
df = df \
    .withColumn("required_age", F.regexp_replace(F.col("required_age"), r"MA 15\+", "15")) \
    .withColumn("required_age", F.regexp_replace(F.col("required_age"), r"21\+", "21")) \
    .withColumn("required_age", F.regexp_replace(F.col("required_age"), r"7\+", "7")) \
    .withColumn('required_age', F.col('required_age').cast('int'))

df \
    .select("required_age") \
    .distinct() \
    .orderBy("required_age") \
    .sort(F.col("required_age")) \
    .show()

+------------+
|required_age|
+------------+
|           0|
|           3|
|           5|
|           6|
|           7|
|           8|
|           9|
|          10|
|          12|
|          13|
|          14|
|          15|
|          16|
|          17|
|          18|
|          20|
|          21|
|          35|
|         180|
+------------+



Certains jeux ont des âges minimaux requis inhabituels (20 et 21 ans, mais surtout 35 et 180 ans).
Comme ils sont très peu nombreux (sept au total), nous ne chercherons pas à corriger une éventuelle erreur et ils seront de ce fait considérés comme des jeux pour adultes.

In [0]:
df \
    .filter(F.col("required_age") > 18) \
    .groupBy("required_age") \
    .count() \
    .orderBy("required_age") \
    .show()

+------------+-----+
|required_age|count|
+------------+-----+
|          20|    1|
|          21|    1|
|          35|    1|
|         180|    4|
+------------+-----+



#### 0.4.4 Nombre de possesseurs d'un jeu

La colonne `owners` est de type `string` et indique un intervalle de nombre de personnes possédant le jeu. L'étendue des valeurs possibles va de 0 à 500 000 000. `IntegerType` est le bon type pour définir cette colonne.

Il reste le choix de la métrique à utiliser pour caractériser les intervalles, par exemple : le minimum, le maximum ou la valeur moyenne.
Le premier, le minimum, imposerait une valeur de 0 pour l'intervalle `[0 .. 20000]` et signifierait que certains jeux peu possédés par des joueurs ne seraient pas possédés du tout.
Le deuxième, le maximum, surestimerait le nombre de possesseurs en permanence.
Le troisième, la moyenne, semble alors le meilleur candidat.

In [0]:
df \
    .select(F.col("owners")) \
    .distinct() \
    .sort(F.col("owners")) \
    .show(truncate=False)

+--------------------------+
|owners                    |
+--------------------------+
|0 .. 20,000               |
|1,000,000 .. 2,000,000    |
|10,000,000 .. 20,000,000  |
|100,000 .. 200,000        |
|2,000,000 .. 5,000,000    |
|20,000 .. 50,000          |
|20,000,000 .. 50,000,000  |
|200,000 .. 500,000        |
|200,000,000 .. 500,000,000|
|5,000,000 .. 10,000,000   |
|50,000 .. 100,000         |
|50,000,000 .. 100,000,000 |
|500,000 .. 1,000,000      |
+--------------------------+



In [0]:
df = df \
    .withColumn(
        "minValueOwners",
        F.when(
            F.col("owners").isNull() | (F.trim(F.col("owners")) == ""),
            None
        ) \
        .otherwise(
            F.regexp_replace(
                F.split(
                    F.translate(F.col("owners"), "[]", ""),  # Suppression des crochets
                    " .. "  # Découpage par rapport à " .. "
                )[0],  # On prend le premier élément
                ",", ""  # Suppression des virgules
            ) \
            .cast("int")
        )
    ).withColumn(
        "maxValueOwners",
        F.when(
            F.col("owners").isNull() | (F.trim(F.col("owners")) == ""),
            None
        ) \
        .otherwise(
            F.regexp_replace(
                F.split(
                    F.translate(F.col("owners"), "[]", ""),
                    " .. "
                )[1],  # On prend le deuxième élément
                ",", ""
            ) \
            .cast("int")
        )
    ) \
    .drop("owners") \
    .withColumn(
        "owners",
        (F.col("minValueOwners") + F.col("maxValueOwners")) / 2.
    ) \
    .drop(*["minValueOwners", "maxValueOwners"])

df \
    .select(F.col("owners")) \
    .distinct() \
    .sort(F.col("owners")) \
    .show(truncate=False)

+---------+
|owners   |
+---------+
|10000.0  |
|35000.0  |
|75000.0  |
|150000.0 |
|350000.0 |
|750000.0 |
|1500000.0|
|3500000.0|
|7500000.0|
|1.5E7    |
|3.5E7    |
|7.5E7    |
|3.5E8    |
+---------+



#### 0.4.5 Ratio d'avis positifs

Nous ajoutons la colonne `nb_reviews` qui définit le nombre d'avis total comme la somme suivante 

$$
\text{nombre d'avis total}= \text{nombre d'avis positifs} + \text{nombre d'avis négatifs}
$$

et la colonne `positive_ratio_PCT` qui définit le pourcentage d'avis positifs relativement au nombre d'avis total :

$$
\text{taux d'avis positifs (\\%) }= \frac{\text{nombre d'avis positifs}}{\text{nombre d'avis positifs} + \text{nombre d'avis négatifs}}
$$

Si le nombre d'avis total est nul, alors nous forçons le taux à 0%.

In [0]:
# Calcul du ratio d'avis positifs
df = df \
    .withColumn("nb_reviews", F.col("positive") + F.col("negative")) \
    .withColumn("positive_ratio_PCT",
        F.when(
            F.col("nb_reviews") > 0,
            F.round(
                (
                    100 * 
                    F.col("positive").cast(DoubleType()) / 
                    F.col("nb_reviews").cast(DoubleType())
                ).cast(DecimalType(5 ,2)),
                2
            )
        ) \
        .otherwise(F.lit(0).cast(DecimalType(5, 2)))
    )
 
df.select("name", "nb_reviews", "positive_ratio_PCT") \
    .orderBy("nb_reviews", ascending=False) \
    .show(10, truncate=False)


+--------------------------------+----------+------------------+
|name                            |nb_reviews|positive_ratio_PCT|
+--------------------------------+----------+------------------+
|Counter-Strike: Global Offensive|6730438   |88.31             |
|PUBG: BATTLEGROUNDS             |2093876   |56.61             |
|Dota 2                          |1852811   |82.84             |
|Grand Theft Auto V              |1442644   |85.21             |
|Tom Clancy's Rainbow Six Siege  |1086157   |86.81             |
|Terraria                        |1037091   |97.84             |
|Team Fortress 2                 |903830    |93.65             |
|Garry's Mod                     |891238    |96.63             |
|Rust                            |844667    |86.72             |
|Left 4 Dead 2                   |660664    |97.45             |
+--------------------------------+----------+------------------+
only showing top 10 rows


#### 0.4.6 Langues

La colonne `languages` est de type `string` et contient les noms des différentes langues disponibles séparés par une virgule suivie d'un espace. Nous la convertissons en `array` de `string`, après avoir découpé la chaîne de caractères initiale en N chaînes (langues).

In [0]:
# Conversion string en liste
df = df \
    .withColumn(
        "languages",
        F.split(
            F.regexp_replace(F.col("languages"), r"\s*,\s*", ","),
            ","
        )
    )
 
df \
    .select("name", "languages") \
    .orderBy("name") \
    .show(3, truncate=False)

+--------------------------------------------+-----------------------------------------+
|name                                        |languages                                |
+--------------------------------------------+-----------------------------------------+
| Fieldrunners 2                             |[English]                                |
| Sakura no Mori † Dreamers 2                |[Simplified Chinese, Traditional Chinese]|
|! That Bastard Is Trying To Steal Our Gold !|[English]                                |
+--------------------------------------------+-----------------------------------------+
only showing top 3 rows


#### 0.4.7 Genres

La même procédure que pour la colonne `languages` est suivie.

In [0]:
# Conversion string en liste
df = df \
    .withColumn(
        "genres",
        F.split(
            F.regexp_replace(F.col("genre"), r"\s*,\s*", ","),
            ","
        )
    ) \
    .drop("genre")
 
df \
    .select("name", "genres") \
    .show(5, truncate=False)

+---------------------------+-------------------------------------+
|name                       |genres                               |
+---------------------------+-------------------------------------+
|Counter-Strike             |[Action]                             |
|ASCENXION                  |[Action, Adventure, Indie]           |
|Crown Trick                |[Adventure, Indie, RPG, Strategy]    |
|Cook, Serve, Delicious! 3?!|[Action, Indie, Simulation, Strategy]|
|细胞战争                   |[Action, Casual, Indie, Simulation]  |
+---------------------------+-------------------------------------+
only showing top 5 rows


### 0.5 Résumé après nettoyage

#### 0.5.1 Données caractéristiques du jeu de données

In [0]:
NB_COLS = len(df.columns)
print(f"Nombre de jeux :     {NB_GAMES}")
print(f"Nombre de colonnes : {NB_COLS}")

Nombre de jeux :     55690
Nombre de colonnes : 20


In [0]:
df.printSchema()

root
 |-- appid: long (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- developer: string (nullable = true)
 |-- languages: array (nullable = true)
 |    |-- element: string (containsNull = false)
 |-- name: string (nullable = true)
 |-- negative: long (nullable = true)
 |-- windows: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- positive: long (nullable = true)
 |-- publisher: string (nullable = true)
 |-- release_date: date (nullable = true)
 |-- required_age: integer (nullable = true)
 |-- price_USD: decimal(6,2) (nullable = true)
 |-- initialprice_USD: decimal(6,2) (nullable = true)
 |-- discount_PCT: decimal(5,2) (nullable = true)
 |-- owners: double (nullable = true)
 |-- nb_reviews: long (nullable = true)
 |-- positive_ratio_PCT: decimal(6,2) (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = false)



#### 0.5.2 Valeurs manquantes

In [0]:
# Colonnes par type
string_cols = [f.name for f in df.schema.fields if str(f.dataType) == "StringType()"]
array_cols = [f.name for f in df.schema.fields if str(f.dataType).startswith("ArrayType")]
other_cols = [f.name for f in df.schema.fields if f.name not in string_cols + array_cols]

# Construction des expressions de comptage
exprs = []

# Colonnes classiques : on compte les NULLs
for c in other_cols:
    exprs.append(
        F.count(
            F.when(F.col(c).isNull(), 1)
        ).alias(c)
    )

# Colonnes string : on compte les NULLs ou les strings vides
for c in string_cols:
    exprs.append(
        F.count(
            F.when(F.col(c).isNull() | (F.col(c) == ""), 1)
        ).alias(c)
    )

# Colonnes array<string> : on compte les NULLs, les tableaux vides, ou les tableaux ne contenant que des strings vides
for c in array_cols:
    exprs.append(
        F.round(
            100 * F.count(
                F.when(
                    F.col(c).isNull()
                    | (F.size(F.col(c)) == 0)
                    | (F.size(F.col(c)) == F.size(F.filter(F.col(c), lambda x: x == ""))),
                    1
                )
            ) / NB_GAMES,
            2
        ) \
        .alias(c)
    )

missing_df = df \
    .select(exprs) \
    .select([(100 * F.col(c) / NB_GAMES).alias(c) for c in df.columns]) \
    .toPandas() \
    .T \
    .sort_values(by=0, ascending=False) \
    .rename(columns={0: "Nb NULLs (%)"})

print(missing_df[missing_df["Nb NULLs (%)"] > 0])

              Nb NULLs (%)
publisher         0.237026
developer         0.226252
release_date      0.177770
categories        0.003124
genres            0.000521
languages         0.000036


Seulement six colonnes contiennent des valeurs manquantes et avec un taux très faible (inférieur à 1%).
Nous maintenons ces lignes dans le jeu de données.
Elles seront signalées ou supprimées au cas par cas pendant l'étude.

#### 0.5.3 Statistiques descriptives

In [0]:
numeric_cols = [
    f.name
    for f in df.schema.fields
    if (
        isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType)) and
        f.name != "appid"
    )
]

df \
    .select(*numeric_cols) \
    .summary() \
    .select(
        "summary",
        *[
            F.when(
                F.col("summary").isin(["mean", "stddev"]),
                F.round(F.col(c).cast(DoubleType()), 2).cast("string")
            ).otherwise(F.col(c)).alias(c)
            for c in numeric_cols
        ]
    ) \
    .show(truncate=False)

+-------+--------+--------+------------+---------+----------------+------------+----------+----------+------------------+
|summary|negative|positive|required_age|price_USD|initialprice_USD|discount_PCT|owners    |nb_reviews|positive_ratio_PCT|
+-------+--------+--------+------------+---------+----------------+------------+----------+----------+------------------+
|count  |55690   |55690   |55690       |55690    |55690           |55690       |55690     |55690     |55690             |
|mean   |241.81  |1470.8  |0.2         |7.73     |7.98            |2.6         |133513.56 |1712.61   |73.44             |
|stddev |5765.46 |30983.01|2.3         |10.93    |11.05           |12.89       |1849924.59|35687.92  |24.52             |
|min    |0       |0       |0           |0.00     |0.00            |0.00        |10000.0   |0         |0.00              |
|25%    |1       |5       |0           |1.34     |1.99            |0.00        |10000.0   |7         |60.56             |
|50%    |6       |19    

---

## 1. Analyse macroscopique

Cette première partie adopte une vue d'ensemble du catalogue Steam, sans segmentation préalable. L'objectif est de dresser le portrait global du marché : qui publie, à quel rythme, à quel prix, dans quelles langues et pour quel public. Ces repères macroscopiques constituent la toile de fond sur laquelle s'appuieront les analyses plus fines des parties suivantes.

Nous y examinons successivement les éditeurs les plus prolifiques, les jeux les mieux notés, l'évolution du nombre de sorties dans le temps (notamment pendant la période Covid), la distribution des prix et des remises, les langues les plus représentées et, enfin, la part des jeux soumis à une restriction d'âge.

### 1.1 Quel éditeur a sorti le plus de jeux sur Steam ?

In [0]:
# ANCIENNE VERSION : noms d'éditeur distinct
#top10_publishers = df \
#    .filter((F.col('publisher') != '') & (F.col('publisher').isNotNull())) \
#    .select('publisher') \
#    .groupBy('publisher') \
#    .agg(F.count('*').alias('nb_games')) \
#    .orderBy(F.desc('nb_games')) \
#    .limit(10)

# NOUVELLE VERSION : noms d'éditeur proches
# Liste des suffixes à supprimer (formes juridiques + termes génériques)
publisher_suffixes = (
    r"\s+(entertainment|games?|studios?|interactive|"
    r"inc\.?|llc|ltd\.?|corp\.?|corporation|group|co\.?|"
    r"gmbh|sarl|sa|limited)$"
)

top10_publishers = df \
    .filter((F.col('publisher') != '') & (F.col('publisher').isNotNull())) \
    .withColumn(
        "publisher_key",
        F.trim(
            F.regexp_replace(F.lower(F.col("publisher")), publisher_suffixes, "")
        )
    ) \
    .groupBy("publisher_key") \
    .agg(
        F.count("*").alias("nb_games"),
        F.first("publisher").alias("publisher")  # nom original représentatif
    ) \
    .orderBy(F.desc("nb_games")) \
    .limit(10)

display(
    top10_publishers \
        .withColumnsRenamed(
            {
                "nb_games": "Nombre de jeux",
                "publisher": "Editeur"
            }
        )
)

Databricks visualization. Run in Databricks to view.

publisher_key,Nombre de jeux,Editeur
big fish,424,Big Fish Games
8floor,244,8floor
sega,165,SEGA
strategy first,152,Strategy First
square enix,141,Square Enix
choice of,140,Choice of Games
hh-games,133,HH-Games
sekai project,132,Sekai Project
ubisoft,132,Ubisoft
laush,126,Laush Studio


**Big Fish Games** est l'éditeur qui a sorti le plus de jeux sur Steam, avec **424 jeux**, loin devant 8floor (244) et SEGA (165). Ubisoft figure dans ce top 10, en huitième position avec 132 jeux.

### 1.2 Quels sont les jeux les mieux notés ?

#### Choix du seuil de filtrage

Un jeu peut atteindre un taux d'avis positifs très élevé (proche de 100%) avec seulement quelques dizaines d'avis émis par un public restreint et acquis. Inclure ces titres dans un classement « jeux les mieux notés » fausserait la lecture : un ratio sur 50 avis n'a pas la même portée qu'un ratio sur 50 000.

Nous appliquons donc un seuil minimum sur le nombre total d'avis (`nb_reviews >= 10000`). Ce seuil est un **choix empirique** assumé, justifié par les ordres de grandeur observés dans le dataset :
- la médiane du nombre d'avis sur l'ensemble du catalogue est de quelques dizaines seulement,
- 10 000 avis correspond à un niveau de visibilité élevé (au-delà du P90 du catalogue), sélectionnant les titres pour lesquels la note traduit un consensus large,
- le jury obtenu (~quelques centaines de jeux) reste statistiquement significatif et exploitable.

D'autres seuils raisonnables (5 000 ou 20 000) auraient mené à un classement comparable dans son esprit, en ne déplaçant les rangs qu'à la marge.

In [0]:
top10_games_per_positive_ratings = df \
    .filter(F.col("nb_reviews") >= 10000) \
    .select(
        'name',
        'positive_ratio_PCT',
        #'positive',
        #'negative',
        #'nb_reviews'
    ) \
    .orderBy(F.desc('positive_ratio_PCT'), F.desc("nb_reviews")) \
    .limit(10)

display(
    top10_games_per_positive_ratings \
        .withColumnsRenamed(
            {
                "name": "Nom du jeu",
                "positive_ratio_PCT": "Ratio d'avis positifs (%)",
                #"positive": "Nombre d'avis positifs",
                #"negative": "Nombre d'avis négatifs",
                #"nb_reviews": "Nombre d'avis"
            }
        )
)

Nom du jeu,Ratio d'avis positifs (%)
Aseprite,99.33
A Short Hike,99.26
Senren＊Banka,99.21
People Playground,98.86
Portal 2,98.78
Vampire Survivors,98.77
The Room 4: Old Sins,98.75
The Henry Stickmin Collection,98.72
ULTRAKILL,98.65
Hades,98.60


Databricks visualization. Run in Databricks to view.

Le jeu le mieux noté est **Aseprite** avec **99,33%** d'avis positifs, suivi de **A Short Hike** (99,26%) et **Senren＊Banka** (99,21%).

Le tri secondaire sur le nombre d'avis fait toutefois ressortir des titres très populaires *et* très bien notés : **Portal 2** (98,78% sur plus de 309 000 avis), **Hades** (98,60% sur près de 203 000 avis) et **Vampire Survivors** (98,77% sur près de 132 000 avis). Ces jeux sont plus représentatifs d'un succès critique de masse, là où les trois premiers atteignent un ratio légèrement supérieur mais sur des volumes d'avis bien plus faibles (environ 10 000 à 12 000).

### 1.3 Est-ce qu'il y a des années avec plus de sorties ? Par exemple, est-ce qu'il y a eu plus ou moins de sorties pendant le Covid ?

#### Remarque méthodologique — extrapolation 2022

Le dataset s'arrête au 315ᵉ jour de 2022 (11 novembre). Conserver la valeur brute aurait suggéré à tort une chute de la production en fin de période. L'extrapolation `× 365 / 315` ramène le compte de 2022 à une année pleine et permet de comparer 2022 aux années précédentes sans introduire ce biais visuel.

Cette extrapolation suppose une **production uniforme sur l'année**, hypothèse que la section 4.4.1 vient nuancer (saisonnalité réelle, avec un pic en octobre et un creux en décembre). Comme le pic d'octobre est déjà capté par les 315 jours observés alors que le creux de fin d'année ne l'est pas, l'extrapolation **surestime probablement le total réel**. L'écart reste néanmoins de l'ordre de quelques pourcents, sans impact sur l'allure de la courbe ni sur le constat de maturation du marché.

In [0]:
nb_games_per_year = df \
    .filter(F.col("release_date").isNotNull()) \
    .withColumn("release_year", F.year(F.col("release_date"))) \
    .groupBy("release_year") \
    .count() \
    .withColumnRenamed("count", "nb_games") \
    .orderBy("release_year")

display(nb_games_per_year)

release_year,nb_games
1997,2
1998,1
1999,3
2000,2
2001,4
2002,1
2003,3
2004,6
2005,6
2006,61


In [0]:
stats_2022 = df \
    .filter(F.year(F.col("release_date")) == 2022) \
    .agg(
        F.count("*").alias("nb_games"),
        F.max(F.dayofyear(F.col("release_date"))).alias("max_day")
    ) \
    .collect()[0]

nb_games_per_year_in_2022 = stats_2022["nb_games"]
max_day_of_year_in_2022 = stats_2022["max_day"]
estimated_nb_games_per_year_in_2022 = int(nb_games_per_year_in_2022 * 365. / max_day_of_year_in_2022 + 0.5)

print(f"Number of games released in 2022 (incomplete year): {nb_games_per_year_in_2022}")
print(f"Max day of year in 2022: {max_day_of_year_in_2022}")
print(f"Extrapolated number of games released in 2022 (complete year): {estimated_nb_games_per_year_in_2022 }")

nb_games_per_year_corrected = nb_games_per_year \
    .withColumn(
        "Nombre de jeux",
        F.when(
            F.col("release_year") == 2022,
            estimated_nb_games_per_year_in_2022
        ) \
        .otherwise(F.col("nb_games"))
    ) \
    .withColumnRenamed("release_year", "Année") \
    .withColumn(
        "Covid-19",
        F.when(
            F.col("Année").between(2020, 2021),
            1
        )
        .otherwise(0)
    ) \
    .drop("nb_games")

display(nb_games_per_year_corrected)

Number of games released in 2022 (incomplete year): 7455
Max day of year in 2022: 315
Extrapolated number of games released in 2022 (complete year): 8638


Databricks visualization. Run in Databricks to view.

Année,Nombre de jeux,Covid-19
1997,2,0
1998,1,0
1999,3,0
2000,2,0
2001,4,0
2002,1,0
2003,3,0
2004,6,0
2005,6,0
2006,61,0


Le nombre de sorties connaît une **très forte croissance entre 2013 et 2018**, passant de 471 à 7 678 jeux, soit une multiplication par plus de 16 en cinq ans. À partir de 2019, la tendance reste haussière mais la **pente s'infléchit nettement** : on observe même un léger recul en 2019 (6 968) avant une reprise jusqu'à un sommet en **2021 (8 823 jeux)**. L'année 2022, incomplète dans le dataset (arrêtée au 315ᵉ jour), a été extrapolée à environ 8 638 jeux pour une année pleine, soit un niveau comparable à 2021. La phase d'expansion explosive laisse donc place, à partir de 2019, à une croissance ralentie qui semble traduire une **maturation du marché** plutôt qu'une poursuite du décollage.

Concernant le Covid, ce ralentissement de la pente **ne s'explique pas par la pandémie** : il s'amorce dès 2019, avant la crise. Sur la période 2019-2021, les sorties continuent au contraire de progresser (6 968 → 8 305 → 8 823), ce qui indique que la production de jeux n'a pas souffert des confinements et a même pu bénéficier du contexte (hausse du temps de jeu). L'inflexion observée relève donc d'une dynamique de marché de fond, indépendante de l'épisode sanitaire.

### 1.4 Quelle est la distribution des prix ? Est-ce qu'il y a beaucoup de jeux avec une remise ?

Répartition jeux payants vs gratuits

In [0]:
paid_vs_free_count_df = (
    df.filter(F.col("price_USD").isNotNull())
      .withColumn(
          "Modèle",
          F.when(F.col("price_USD") > 0, "Jeu payant").otherwise("Jeu gratuit")
      )
      .groupBy("Modèle")
      .count()
      .transform(lambda d: d.crossJoin(d.agg(F.sum("count").alias("tot"))))
      .withColumn(
          "Nombre de jeux (%)",
          F.round(F.col("count") / F.col("tot") * 100, 2)
      )
      .select("Modèle", "Nombre de jeux (%)")
)

display(paid_vs_free_count_df)

Databricks visualization. Run in Databricks to view.

Modèle,Nombre de jeux (%)
Jeu gratuit,13.97
Jeu payant,86.03


Distribution des prix (jeux gratuits exclus) après suppression des valeurs aberrantes

In [0]:
prices = df \
    .filter(
        (F.col("price_USD").isNotNull()) & (F.col("price_USD") > 0)
    ) \
    .select(F.col("price_USD"))

stats = prices.select(
    F.expr("percentile(price_USD, 0.25)").alias("q1"),
    F.expr("percentile(price_USD, 0.75)").alias("q3")
).first()

q1_prices, q3_prices = stats["q1"], stats["q3"]
iqr_prices = q3_prices - q1_prices
#lower_limit, upper_limit = q1_prices - 1.5 * iqr_prices, q3_prices + 1.5 * iqr_prices
lower_limit, upper_limit = 0, 100
print(f"Q1 = {q1_prices}")
print(f"Q3 = {q3_prices}")
print(f"IQR = {iqr_prices}")
print(f"lower_limit = {lower_limit}")
print(f"upper_limit = {upper_limit}")

prices_without_outliers = prices \
    .filter(
        F.col("price_USD").between(lower_limit, upper_limit)
    ) \
    .orderBy(F.col("price_USD").asc())

display(
    prices_without_outliers \
        .withColumnRenamed("price_USD", "prix (USD)")
)

Q1 = 2.99
Q3 = 10.0
IQR = 7.01
lower_limit = 0
upper_limit = 100


Databricks visualization. Run in Databricks to view.

prix (USD)
0.28
0.28
0.28
0.28
0.28
0.28
0.28
0.28
0.28
0.28


Jeux avec remise (jeux gratuits exclus)

In [0]:
discount_or_not_df = df \
    .filter(
        (F.col("discount_PCT").isNotNull()) & 
        (F.col("price_USD") > 0)
        ) \
    .withColumn(
        "Remise",
        F.when(F.col("discount_PCT") > 0, "Oui").otherwise("Non")
    ) \
    .groupBy("Remise") \
    .count() \
    .transform(lambda d: d.crossJoin(d.agg(F.sum("count").alias("tot")))) \
    .withColumn(
        "Nombre de jeux (%)",
        F.round(F.col("count") / F.col("tot") * 100, 2)
    ) \
    .select("Remise", "Nombre de jeux (%)")

display(discount_or_not_df)

Databricks visualization. Run in Databricks to view.

Remise,Nombre de jeux (%)
Oui,5.26
Non,94.74


Le catalogue est majoritairement payant : **86,0%** de jeux payants contre 14,0% de gratuits. Pour les jeux payants, les prix sont concentrés sur des montants bas (médiane et quartiles serrés : Q1 = 2,99 USD, Q3 = 10,00 USD), confirmant un marché dominé par les petits prix. Enfin, les remises sont rares : seuls **5,26%** des jeux payants affichent une réduction.

### 1.5 Quels sont les langues les plus représentées ?

In [0]:
c = "languages"
top10_languages = df \
    .filter(
        (F.col(c).isNotNull())
        & (F.size(F.col(c)) > 0)
        & (F.size(F.col(c)) != F.size(F.filter(F.col(c), lambda x: x == "")))
    ) \
    .withColumn(
        "language",
        F.explode(F.col(c))
    ) \
    .withColumn("language", F.trim(F.col("language"))) \
    .groupBy("language") \
    .count() \
    .withColumnRenamed("count", "Nombre de jeux") \
    .withColumnRenamed("language", "Langue") \
    .orderBy(F.col("Nombre de jeux").desc()) \
    .limit(10)

display(top10_languages)

Databricks visualization. Run in Databricks to view.

Langue,Nombre de jeux
English,55116
German,14019
French,13426
Russian,12922
Simplified Chinese,12782
Spanish - Spain,12233
Japanese,10368
Italian,9304
Portuguese - Brazil,6750
Korean,6600


L'**anglais** domine très largement avec **55 116 jeux**, soit la quasi-totalité du catalogue. Suivent, loin derrière, l'**allemand** (14 019), le **français** (13 426), le **russe** (12 922) et le **chinois simplifié** (12 782). Le français se positionne ainsi troisième, ce qui en fait une langue de localisation pertinente pour Ubisoft.

### 1.6 Est-ce qu'il y a beaucoup de jeux interdits aux enfants de moins de 16 ans ? Aux moins de 18 ans ?

In [0]:
c = "required_age"
required_age_df = df \
    .withColumn(
        "catégorie",
        F.when(F.col(c).isNull(), F.lit("Valeur manquante"))
        .when(F.col(c) >= 18, F.lit("PEGI 18"))
        .when(F.col(c) >= 16, F.lit("PEGI 16"))
        .when(F.col(c) >= 12, F.lit("PEGI 12"))
        .when(F.col(c) >= 7, F.lit("PEGI 7"))
        .when(F.col(c) == 0, F.lit("Non renseigné ou tous publics"))
        .otherwise(F.lit("PEGI 3"))
    )

required_age_print = required_age_df \
    .groupBy("catégorie") \
    .count() \
    .transform(lambda d: d.crossJoin(d.agg(F.sum("count").alias("tot")))) \
    .withColumn(
        "Nombre de jeux (%)",
        F.round(F.col("count") / F.col("tot") * 100, 2)
    ) \
    .withColumnRenamed("count", "Nombre de jeux") \
    .sort(F.desc("Nombre de jeux (%)")) \
    .select("catégorie", "Nombre de jeux","Nombre de jeux (%)")

required_age_print.show(truncate=False)
display(required_age_print.select("catégorie", "Nombre de jeux (%)"))


+-----------------------------+--------------+------------------+
|catégorie                    |Nombre de jeux|Nombre de jeux (%)|
+-----------------------------+--------------+------------------+
|Non renseigné ou tous publics|55029         |98.81             |
|PEGI 12                      |333           |0.6               |
|PEGI 18                      |230           |0.41              |
|PEGI 16                      |76            |0.14              |
|PEGI 7                       |14            |0.03              |
|PEGI 3                       |8             |0.01              |
+-----------------------------+--------------+------------------+



catégorie,Nombre de jeux (%)
Non renseigné ou tous publics,98.81
PEGI 12,0.6
PEGI 18,0.41
PEGI 16,0.14
PEGI 7,0.03
PEGI 3,0.01


La quasi-totalité (98,81%) des jeux :
- soit n'ont pas d'âge minimum requis,
- soit l'âge minimum n'a pas été défini et une valeur par défaut (nulle) est restée.

Cette absence d'information semble indiquer que Steam n'oblige pas les éditeurs à mentionner un âge minimum requis. Il est possible que des indications de la maturité nécessaire existent (par exemple via les tags ou les descriptions) par ailleurs ou qu'un contrôle de l'âge de l'utilisateur se fasse par un autre mécanisme.

Même si cela est peu représentatif, à hauteur de 1,19% des jeux présents, nous montrons ci-dessous la répartition des catégories d'âge minimum requis où une valeur non nulle est indiquée.

In [0]:
display(required_age_df.filter(F.col("catégorie") != "Non renseigné ou tous publics").select("catégorie"))

Databricks visualization. Run in Databricks to view.

catégorie
PEGI 12
PEGI 12
PEGI 12
PEGI 18
PEGI 18
PEGI 12
PEGI 18
PEGI 16
PEGI 18
PEGI 16


---

**98,81%** des jeux n'ont aucun âge minimum renseigné. Parmi le 1,19% restant, les jeux PEGI 18 ne représentent que **0,41%** du catalogue (230 jeux) et les PEGI 16 seulement **0,14%** (76 jeux). Les restrictions d'âge formelles sont donc quasi inexistantes sur Steam, qui ne semble pas imposer aux éditeurs de renseigner ce champ.

### Bilan partiel — Analyse macroscopique

Cette première partie dresse le portrait global du marché Steam et fait ressortir plusieurs constats utiles à la stratégie d'Ubisoft.

Sur le plan des **éditeurs**, le catalogue est dominé en volume par des acteurs spécialisés dans la production de masse (Big Fish Games, 424 jeux ; 8floor, 244), plutôt que par les grands studios AAA (Rockstar Games, Sony, Electronic Arts, Microsoft, etc.). Ubisoft figure néanmoins dans le top 10 (132 jeux, 8ᵉ position), ce qui témoigne d'une présence déjà solide mais sans logique de saturation du catalogue.

L'analyse des **notes** montre qu'un excellent ratio d'avis positifs (supérieur à 98%) reste atteignable même sur de très gros volumes : Portal 2, Hades ou Vampire Survivors combinent succès critique et popularité de masse, là où les tout premiers du classement atteignent un ratio légèrement supérieur mais sur des volumes d'avis bien plus faibles. La qualité perçue n'est donc pas réservée aux titres confidentiels.

L'évolution des **sorties** révèle une croissance explosive entre 2013 et 2018 (×16 en cinq ans), suivie à partir de 2019 d'une croissance ralentie qui traduit une maturation du marché plutôt qu'un essoufflement. Le Covid n'a pas freiné la production : les sorties progressent sur 2019-2021, signe d'un marché résilient.

Côté **modèle économique**, le catalogue est très majoritairement payant (86%), avec des prix concentrés sur des montants bas (médiane proche de 5 USD, Q3 à 10 USD). Les remises sont rares dans l'instantané observé (5,3%), hors périodes de soldes Steam.

L'analyse des **langues** confirme l'hégémonie de l'anglais (quasi-totalité du catalogue), suivi de l'allemand, du français (3ᵉ rang) et du russe. Le français apparaît ainsi comme une langue de localisation pertinente.

Enfin, les **restrictions d'âge** sont quasi inexistantes : 98,8% des jeux n'ont aucun âge minimum renseigné, Steam n'imposant pas ce champ aux éditeurs.

En synthèse, Steam est un marché massif, mature et fortement concurrentiel, où le bas prix est la norme et où la qualité de réception ne dépend pas du volume. Pour Ubisoft, le défi se situe moins dans le volume publié que dans la différenciation qualitative, abordée dans les parties suivantes.

## 2. Analyse des genres

Après la vue d'ensemble, nous descendons d'un niveau pour analyser le marché par genre. Le genre est un axe structurant pour Ubisoft : il oriente à la fois le positionnement d'un futur jeu, son public cible et son potentiel commercial.

Nous étudions ici quels genres dominent le catalogue, lesquels obtiennent les meilleurs taux d'avis positifs, si les principaux éditeurs ont des genres de prédilection, et enfin quels genres se révèlent les plus rémunérateurs.

### 2.1 Quels sont les genres les plus représentés ?

In [0]:
c = "genres"
genres_df = df \
    .withColumn(
        "Genre",
        F.explode(F.col(c))
    ) \
    .filter(F.col("Genre") != "") \
    .withColumn("Genre", F.trim(F.col("Genre")))

top10_genres = genres_df \
    .groupBy("Genre") \
    .count() \
    .withColumnRenamed("count", "Nombre de jeux") \
    .orderBy(F.col("Nombre de jeux").desc()) \
    .limit(10)

display(top10_genres)

Databricks visualization. Run in Databricks to view.

Genre,Nombre de jeux
Indie,39681
Action,23759
Casual,22086
Adventure,21431
Strategy,10895
Simulation,10836
RPG,9534
Early Access,6145
Free to Play,3393
Sports,2666


Le genre **Indie** domine massivement le catalogue avec **39 681 jeux**, soit près du double du genre suivant. Viennent ensuite **Action** (23 759), **Casual** (22 086) et **Adventure** (21 431), puis un second groupe plus resserré autour de Strategy (10 895), Simulation (10 836) et RPG (9 534). Cette hiérarchie confirme le constat de la partie 1 : Steam est avant tout une plateforme de production indépendante et de jeux accessibles, plutôt qu'un catalogue dominé par les grosses productions.

**Remarque :** Un jeu pouvant porter plusieurs genres, ces effectifs se recoupent (somme supérieure au nombre de jeux).

### 2.2 Est-ce qu'il y a des genres avec un taux d'avis positifs plus élevé ? Avec un taux d'avis négatifs plus élevé ?

Comme en 1.2, nous appliquons le seuil `nb_reviews >= 10000` pour ne retenir que les jeux dont le taux d'avis positifs repose sur un volume statistiquement significatif. Le complément `Nombre de jeux >= 10` après le `groupBy("Genre")` filtre en outre les genres trop peu peuplés pour porter une moyenne crédible.

In [0]:
genres_by_ratings = genres_df \
    .filter(F.col("nb_reviews") >= 10000) \
    .groupBy("Genre") \
    .agg(
        F.round(F.mean("positive_ratio_PCT"), 2).alias("Ratio moyen d'avis positifs (%)"),
        F.round(100 - F.mean("positive_ratio_PCT"), 2).alias("Ratio moyen d'avis négatifs (%)"),
        F.round(F.mean("nb_reviews"), 0).alias("Nombre moyen d'avis"),
        F.count("*").alias("Nombre de jeux"),
    ) \
    .filter(F.col("Nombre de jeux") >= 10)

top10_genres_by_ratings = genres_by_ratings \
    .orderBy("Ratio moyen d'avis positifs (%)", ascending=False) \
    .limit(10)

flop10_genres_by_ratings = genres_by_ratings \
    .orderBy("Ratio moyen d'avis négatifs (%)", ascending=False) \
    .limit(10)

display(top10_genres_by_ratings.select("Genre", "Ratio moyen d'avis positifs (%)"))

Databricks visualization. Run in Databricks to view.

Genre,Ratio moyen d'avis positifs (%)
Indie,88.89
Casual,88.51
Racing,86.51
Adventure,85.97
Simulation,85.94
Strategy,85.70
Action,84.69
RPG,84.13
Early Access,82.26
Sports,79.03


In [0]:
display(flop10_genres_by_ratings.select("Genre", "Ratio moyen d'avis négatifs (%)"))

Databricks visualization. Run in Databricks to view.

Genre,Ratio moyen d'avis négatifs (%)
Massively Multiplayer,24.84
Free to Play,21.17
Sports,20.97
Early Access,17.74
RPG,15.87
Action,15.31
Strategy,14.30
Simulation,14.06
Adventure,14.03
Racing,13.49


Certains genres sont mieux notés que d'autres, mais les écarts restent modérés (jeux avec au moins 10 000 avis). Les genres les **mieux notés** sont **Indie (88,9%)** et **Casual (88,5%)** d'avis positifs, suivis de Racing (86,5%) et Adventure (86,0%). À l'opposé, les genres au **taux d'avis négatifs le plus élevé** sont **Massively Multiplayer (24,8%)**, **Free to Play (21,2%)** et **Sports (21,0%)**. La tendance est cohérente : les jeux indépendants et casual, souvent ciblés et peu chers, satisfont mieux leur public, tandis que les jeux multijoueurs et free-to-play, plus exposés (monétisation, serveurs, attentes élevées), concentrent davantage d'insatisfaction.

### 2.3 Est-ce que certains éditeurs ont des genres favoris ?

Pour cette question, nous avons fait le choix de limiter les éditeurs possibles à ceux trouvés au préalable à la question 1.1 (DataFrame `top10_publishers`), c'est-à-dire les éditeurs ayant sorti le plus de jeux.

**Remarque :** Le `join` est effectué sur le nom littéral de l'éditeur, sans normalisation. Les chiffres ci-dessous reposent donc sur les noms non consolidés (ex. « Ubisoft » au sens strict, hors variantes type « Ubisoft Entertainment »). L'écart avec le top 10 consolidé du §1.1 est marginal et ne change pas les ordres de grandeur.

In [0]:
fav_genres_from_publishers = genres_df \
    .join(top10_publishers, on="publisher", how="inner") \
    .groupBy("publisher", "Genre") \
    .count() \
    .withColumnsRenamed(
        {
            "count": "Nombre de jeux",
            "publisher": "Editeur"
        }
    ) \
    .orderBy("publisher", "Nombre de jeux", ascending=False)

display(fav_genres_from_publishers)

Databricks visualization. Run in Databricks to view.

Editeur,Genre,Nombre de jeux
Ubisoft,Action,70
Ubisoft,Adventure,45
Ubisoft,Strategy,22
Ubisoft,RPG,19
Ubisoft,Simulation,18
Ubisoft,Racing,14
Ubisoft,Casual,11
Ubisoft,Indie,7
Ubisoft,Free to Play,6
Ubisoft,Sports,5


Les profils sont nettement marqués selon les éditeurs du top 10. **Ubisoft** est très orienté **Action (70 jeux)** et **Adventure (45)**, conformément à son catalogue AAA. **Square Enix** se partage entre **Action (75)** et **RPG (71)**, signature des grands J-RPG. **Strategy First** est logiquement centré sur la **Stratégie (52)**, et **Sekai Project** sur le **Casual (99)** et l'**Indie (88)**, cohérent avec son positionnement de visual novels. Chaque éditeur possède donc une identité de genre claire, reflet de son modèle éditorial.

### 2.4 Quels sont les genres les plus rémunérateurs ?

Pour trouver les genres les plus rémunérateurs, nous avons décidé de calculer les chiffres d'affaires (CA).

#### Chiffre d'affaires

Pour un jeu, le CA est tel que 

\\[
    \text{CA} \quad = \quad \text{prix de vente} \quad \times \quad \text{nombre de possesseurs}
\\]

Ensuite, nous faisons les agrégations par genre pour obtenir :
- le CA total par genre,
- le CA moyen par genre et par jeu.

#### Remarques méthodologiques

Le CA obtenu est une **estimation indicative**, à lire comme un ordre de grandeur pour **classer les genres entre eux**, non comme un montant exact. Plusieurs limites sont à garder en tête :

- **Multi-genres :** un jeu portant plusieurs genres est compté dans chacun, ce qui gonfle artificiellement les CA total et moyen par genre.
- **`owners` ≠ ventes payantes :** sont inclus les jeux obtenus via bundles, offerts ou en forte promotion. Le CA réel est donc **surestimé** par rapport à `prix catalogue × possesseurs`.
- **Intervalle large d'`owners` :** la valeur dérive d'un intervalle (ex. `[20M ; 50M]` → moyenne 35M), introduisant une incertitude au niveau du jeu, lissée par l'agrégation.
- **Modèles free-to-play :** fréquents en *Massively Multiplayer* et *Free to Play*, leurs revenus passent par les microtransactions, abonnements ou cosmétiques, non capturés par `price_USD`. Le CA est ici fortement **sous-estimé**.

Malgré ces limites, le proxy reste pertinent pour comparer les genres à méthode constante.

In [0]:
revenue_by_genre_df = genres_df \
    .withColumn("revenue_USD", F.col("price_USD") * F.col("owners")) \
    .groupBy("Genre") \
    .agg(
        (F.round(F.sum("revenue_USD") / 1e9, 2)).alias("CA total (Md USD)"),
        (F.round(F.mean("revenue_USD") / 1e6, 2)).alias("CA moyen (M USD)")
    ) \
    .select("Genre", "CA total (Md USD)", "CA moyen (M USD)")

print("Top 10 du chiffre d'affaires total par genre :")
top10_total_revenue_by_genre_df = revenue_by_genre_df \
    .select("Genre", "CA total (Md USD)") \
    .sort(F.desc("CA total (Md USD)")) \
    .limit(10)
top10_total_revenue_by_genre_df.show(truncate=False)

print("Top 10 du chiffre d'affaires moyen de chaque jeu par genre :")
top10_mean_revenue_by_genre_df = revenue_by_genre_df \
    .select("Genre", "CA moyen (M USD)") \
    .sort(F.desc("CA moyen (M USD)")) \
    .limit(10)
top10_mean_revenue_by_genre_df.show(truncate=False)

display(top10_total_revenue_by_genre_df)
display(top10_mean_revenue_by_genre_df)

Top 10 du chiffre d'affaires total par genre :
+---------------------+-----------------+
|Genre                |CA total (Md USD)|
+---------------------+-----------------+
|Action               |58.76            |
|Adventure            |37.25            |
|Indie                |32.35            |
|RPG                  |27.17            |
|Strategy             |20.15            |
|Simulation           |18.77            |
|Casual               |8.08             |
|Massively Multiplayer|5.93             |
|Early Access         |5.46             |
|Sports               |3.15             |
+---------------------+-----------------+

Top 10 du chiffre d'affaires moyen de chaque jeu par genre :
+---------------------+----------------+
|Genre                |CA moyen (M USD)|
+---------------------+----------------+
|Massively Multiplayer|4.06            |
|RPG                  |2.85            |
|Action               |2.47            |
|Strategy             |1.85            |
|Adventure      

Databricks visualization. Run in Databricks to view.

Genre,CA total (Md USD)
Action,58.76
Adventure,37.25
Indie,32.35
RPG,27.17
Strategy,20.15
Simulation,18.77
Casual,8.08
Massively Multiplayer,5.93
Early Access,5.46
Sports,3.15


Databricks visualization. Run in Databricks to view.

Genre,CA moyen (M USD)
Massively Multiplayer,4.06
RPG,2.85
Action,2.47
Strategy,1.85
Adventure,1.74
Simulation,1.73
Photo Editing,1.66
Racing,1.26
Web Publishing,1.18
Sports,1.18


Le résultat dépend de la métrique retenue. En **chiffre d'affaires total**, le genre **Action** domine largement (**58,8 Md USD**), devant Adventure (37,3) et Indie (32,4) : ces genres cumulent beaucoup grâce à leur volume de jeux et de joueurs. En **CA moyen par jeu**, le classement change : **Massively Multiplayer** arrive en tête (**4,06 M USD / jeu**), devant RPG (2,85) et Action (2,47). Autrement dit, l'Action est le genre le plus rémunérateur en masse, mais un jeu *MMO* ou *RPG* rapporte en moyenne davantage à l'unité, ce qui en fait des genres à fort potentiel individuel : un arbitrage volume/rentabilité utile à Ubisoft.

### Bilan partiel — Analyse des genres

Cette deuxième partie segmente le marché par genre et fait apparaître une structure claire, utile au positionnement d'un futur jeu Ubisoft.

En **représentation**, le catalogue est largement dominé par le genre Indie (39 681 jeux), suivi d'Action (23 759), Casual (22 086) et Adventure (21 431). Cette hiérarchie prolonge le constat de la partie 1 : Steam est avant tout une plateforme de production indépendante et de jeux accessibles. Les genres "cœur AAA" comme Action restent toutefois très présents, ce qui confirme qu'il existe une place pour des productions à plus fort budget.

Sur la **qualité de réception**, les écarts entre genres restent modérés. Les jeux Indie (88,9%) et Casual (88,5%) obtiennent les meilleurs taux d'avis positifs, tandis que les genres Massively Multiplayer (24,8% d'avis négatifs), Free to Play et Sports concentrent l'insatisfaction. La logique est cohérente : les jeux ciblés et peu chers satisfont mieux leur public, alors que les jeux multijoueurs et free-to-play, plus exposés (monétisation, serveurs, attentes élevées), sont jugés plus durement.

L'analyse des **éditeurs** révèle des identités de genre nettes : Ubisoft est fortement orienté Action et Adventure, Square Enix se partage entre Action et RPG, Strategy First reste centré sur la stratégie et Sekai Project sur le casual et l'indie. Chaque éditeur reflète ainsi son modèle éditorial, ce qui suggère qu'Ubisoft capitalise sur un savoir-faire identifié plutôt que de se disperser.

Enfin, sur la **rentabilité**, le résultat dépend de la métrique. En chiffre d'affaires total, l'Action domine très largement (58,8 Md USD) grâce au volume. Mais en CA moyen par jeu, le Massively Multiplayer (4,06 M USD) et le RPG (2,85 M USD) passent devant : ces genres rapportent davantage à l'unité.

En synthèse, l'Action apparaît comme le genre le plus stratégique pour Ubisoft : très représenté, correctement noté, leader en CA total et déjà au cœur de son catalogue. Les genres MMO et RPG offrent quant à eux un potentiel de rentabilité unitaire supérieur, au prix d'une exigence de réception plus forte. Cet arbitrage volume/rentabilité est approfondi dans les parties suivantes.

---

## 3. Analyse des plateformes

**Note terminologique :** par souci de cohérence avec le vocabulaire de Steam, le mot *plateforme* est employé pour désigner Windows, Mac et Linux. Techniquement, il s'agit toutefois de *systèmes d'exploitation* (OS) plutôt que de plateformes au sens des consoles de jeu. 

Le choix des plateformes supportées est une décision technique et stratégique : il conditionne l'audience accessible et le coût de développement. Cette partie mesure la couverture réelle des trois plateformes (Windows, Mac, Linux) sur le catalogue Steam.

Nous regardons d'abord la disponibilité globale des jeux par plateforme, puis si certains genres tendent à privilégier une plateforme plutôt qu'une autre.

### 3.1 Est-ce que la plupart des jeux sont disponibles sur Windows ? Sur Mac ? Sur Linux ?

In [0]:
platforms = ["Windows", "Mac", "Linux"]
plateforms_df = df \
    .agg(
        *[F.round(100 * F.mean(F.col(p.lower()).cast(IntegerType())), 2).alias(p) for p in platforms]
    ) \
    .select(*platforms) \
    .unpivot(
        ids = [],
        values = platforms,
        variableColumnName = "Plateforme",
        valueColumnName = "Disponibilité (%)"
    )

display(plateforms_df)

Databricks visualization. Run in Databricks to view.

Plateforme,Disponibilité (%)
Windows,99.97
Mac,22.93
Linux,15.19


La domination de **Windows est écrasante** : 99,97% des jeux y sont disponibles, soit la quasi-totalité du catalogue. **Mac** n'est supporté que par 22,93% des jeux et **Linux** par seulement 15,19%. Autrement dit, près de huit jeux sur dix ne sont pas jouables sur Mac, et plus de huit sur dix ne le sont pas sur Linux. Windows est donc la plateforme incontournable et de fait le standard de facto sur Steam, le support Mac et Linux restant minoritaire et optionnel.

### 3.2 Est-ce que certains genres ont tendance à être préférentiellement disponible sur certaines plateformes ?

Nous vérifions que, pour tous les genres, les jeux sont bien disponibles sur Windows. Si c'est le cas, nous regarderons l'éventuelle préférence de plateforme pour les genres en excluant Windows, pour ne pas polluer l'analyse.

In [0]:
platforms_by_genre_df = df \
    .withColumn(
        "Genre",
        F.explode(F.col("genres"))
    ) \
    .filter(F.col("Genre") != "") \
    .groupBy("Genre") \
    .agg(
        *[F.round(100 * F.mean(F.col(p.lower()).cast(IntegerType())), 2).alias(p) for p in platforms]
    ) \
    .select("Genre", *platforms)

nb_genres = platforms_by_genre_df.count()
print(f"Number of genres: {nb_genres}\n")
#platforms_by_genre_df.select("Genre", "Windows").orderBy("Windows").show(nb_genres, truncate=False)
display(platforms_by_genre_df.select("Genre", "Windows").orderBy("Windows"))

Number of genres: 28



Genre,Windows
Audio Production,98.97
Design & Illustration,99.75
Utilities,99.85
Massively Multiplayer,99.93
Free to Play,99.94
Racing,99.95
Simulation,99.96
Sports,99.96
Strategy,99.97
Casual,99.98


Mis à part, peut-être, le genre *Production audio* où environ 1% des jeux ne sont pas disponibles sur Windows, le reste des genres est en quasi-totalité disponible sur cette plateforme. Nous analysons alors seulement la répartition de disponibilité des jeux par genre sur Mac et Linux.

In [0]:
linux_or_mac_by_genre_df = platforms_by_genre_df \
    .select("Genre", "Mac", "Linux") \
    .withColumn(
        "Ratio disponibilité Mac (%)",
        F.when(F.col("Mac") + F.col("Linux") < 0.001, 0) \
        .otherwise(F.round(100 * F.col("Mac") / (F.col("Mac") + F.col("Linux")), 2))
        ) \
    .withColumn(
        "Ratio disponibilité Linux (%)",
        F.when(F.col("Mac") + F.col("Linux") < 0.001, 0) \
        .otherwise(F.round(100 - F.col("Ratio disponibilité Mac (%)"), 2))
        )

linux_or_mac_by_genre_df_long = linux_or_mac_by_genre_df \
    .select("Genre", "Ratio disponibilité Mac (%)", "Ratio disponibilité Linux (%)") \
    .withColumnsRenamed(
        {
            "Ratio disponibilité Mac (%)": "Mac",
            "Ratio disponibilité Linux (%)": "Linux"
        }
    ) \
    .unpivot(
        ids = ["Genre"],
        values = ["Mac", "Linux"],
        variableColumnName = "Plateforme",
        valueColumnName = "Ratio disponibilité (%)"
    )

display(linux_or_mac_by_genre_df_long.orderBy("Genre"))

Databricks visualization. Run in Databricks to view.

Genre,Plateforme,Ratio disponibilité (%)
Accounting,Mac,100.0
Accounting,Linux,0.0
Action,Mac,57.46
Action,Linux,42.54
Adventure,Mac,60.41
Adventure,Linux,39.59
Animation & Modeling,Mac,66.07
Animation & Modeling,Linux,33.93
Audio Production,Mac,85.42
Audio Production,Linux,14.58


Première vérification : sur les 28 genres, **tous sont disponibles sur Windows à plus de 98,9%** (le minimum étant Audio Production à 98,97%). Windows n'étant pas discriminant, l'analyse de la préférence de plateforme se fait en le retirant et en comparant uniquement la répartition Mac/Linux.

Sur cette base, **Mac est systématiquement privilégié face à Linux**, quel que soit le genre. Pour les genres de jeux classiques, la répartition est relativement homogène, autour de 58 à 63% pour Mac contre 37 à 42% pour Linux (par exemple Action : 57,46% Mac / 42,54% Linux ; RPG : 59,61% / 40,39% ; Strategy : 62,2% / 37,8%). Linux ne dépasse jamais Mac.

Les écarts les plus marqués concernent surtout les genres **utilitaires et créatifs**, encore plus orientés Mac : Video Production (82,85% Mac), Audio Production (85,42%), Photo Editing (80%) ou Software Training (69,22%). Ce constat est cohérent avec l'usage professionnel répandu de macOS pour la création audio/vidéo. À l'inverse, aucun genre ne montre de préférence nette pour Linux. Quelques genres (Movie, Accounting) affichent des valeurs extrêmes (0% ou 100%) mais reposent sur de très faibles effectifs et ne sont pas significatifs.

En synthèse : la plateforme n'influence quasiment pas la disponibilité d'un genre sur Windows (universelle), et lorsqu'un éditeur élargit le support au-delà de Windows, il privilégie nettement Mac plutôt que Linux, en particulier pour les logiciels à vocation créative.

### Bilan partiel — Analyse des plateformes

Cette partie confirme une réalité sans ambiguïté : sur Steam, **Windows est le standard de facto**. La plateforme est supportée par 99,97% des jeux, contre seulement 22,93% pour Mac et 15,19% pour Linux. Le support multiplateforme reste donc minoritaire et optionnel.

Tous les genres étant disponibles sur Windows à plus de 98,9%, la plateforme n'est pas un facteur discriminant. En revanche, dès qu'un jeu est porté au-delà de Windows, **Mac est systématiquement privilégié face à Linux** (environ 60% contre 40% pour les genres classiques), avec une orientation Mac encore plus marquée pour les logiciels créatifs (Video Production, Audio Production, Photo Editing).

Pour Ubisoft, l'enseignement est direct : prioriser un développement Windows est indispensable, un portage Mac constitue l'extension la plus pertinente si un multiplateforme est envisagé, et Linux ne se justifie que pour des cas ciblés.

---

## 4. Pour aller plus loin

Les parties précédentes répondent aux questions proposées dans le cadre du projet. Cette dernière partie prolonge l'analyse par des croisements de variables visant à éclairer plus directement la question centrale d'Ubisoft : quels facteurs influencent réellement la popularité et la réception d'un jeu ?

Nous explorons quatre axes : le croisement prix / qualité / popularité, la comparaison entre auto-édition et édition tierce, le lien entre effort de localisation et visibilité, et enfin la saisonnalité des sorties.

### 4.1 Croisement prix / qualité / popularité

#### 4.1.1 Existe-t-il une corrélation entre le prix d'un jeu et son ratio d'avis positifs ? Les jeux plus chers sont-ils mieux notés, ou est-ce l'inverse ?

##### Remarque méthodologique — limite du coefficient de corrélation

Le coefficient de Pearson utilisé ici mesure une relation **linéaire** entre deux variables. C'est un choix simple et lisible, mais il a une limite dans notre cas : la distribution des prix est fortement **asymétrique** (médiane proche de 5 USD, Q3 à 10 USD, longue queue au-delà de 60 USD), ce qui peut affaiblir la sensibilité du coefficient à une éventuelle relation non linéaire.

Toutefois, la conclusion ne repose pas uniquement sur ce coefficient : l'analyse par tranches de prix montre que le taux moyen d'avis positifs reste compris entre 75,4% et 78,2%, soit une amplitude de moins de 3 points. Ce résultat ne suppose aucune hypothèse de linéarité et confirme l'absence de lien notable entre prix et satisfaction. Le constat est donc robuste par triangulation.

In [0]:
# Les jeux gratuits et sans avis positifs sont exclus
paid_games_with_positive_ratings_df = df \
    .filter(F.col("price_USD") > 0) \
    .filter(F.col("positive") > 0)

mean_positive_ratio_PCT = paid_games_with_positive_ratings_df \
    .agg(
        F.round(F.mean("positive_ratio_PCT"), 2).alias("positive_ratio_PCT")
    ) \
    .collect()[0]["positive_ratio_PCT"]

print(f"Taux moyen d'avis positifs pour tous les jeux payants : {mean_positive_ratio_PCT:.1f}%")

# Calcul de la corrélation entre prix et taux d'avis positifs
correlation_price_and_positive_ratio = paid_games_with_positive_ratings_df \
    .stat.corr("price_USD", "positive_ratio_PCT")

print(f"Corrélation entre prix et ratio d'avis positifs : {correlation_price_and_positive_ratio:.4f}")

# Analyse par tranche de prix
positive_ratio_by_price_range = paid_games_with_positive_ratings_df \
    .withColumn(
        "Indice de tranche",
        F.when(F.col("price_USD") < 5, 1)
        .when(F.col("price_USD") < 15, 2)
        .when(F.col("price_USD") < 30, 3)
        .when(F.col("price_USD") < 60, 4)
        .otherwise(5)
    ) \
    .groupBy(F.col("Indice de tranche")) \
    .agg(
        F.round(F.mean("positive_ratio_PCT"), 1).alias("Taux moyen d'avis positifs (%)"),
        F.count("*").alias("Nombre de jeux")
    ) \
    .withColumn(
        "Tranche de prix (USD)",
        F.when(F.col("Indice de tranche") == 1, "< 5")
        .when(F.col("Indice de tranche") == 2, "5 - 15")
        .when(F.col("Indice de tranche") == 3, "15 - 30")
        .when(F.col("Indice de tranche") == 4, "30 - 60")
        .otherwise("> 60")
    ) \
    .orderBy(F.col("Indice de tranche")) \
    .drop(F.col("Indice de tranche"))

display(
    positive_ratio_by_price_range \
        .select(
            "Tranche de prix (USD)",
            "Taux moyen d'avis positifs (%)",
            "Nombre de jeux"
        )
)

Taux moyen d'avis positifs pour tous les jeux payants : 76.8%
Corrélation entre prix et ratio d'avis positifs : 0.0375


Databricks visualization. Run in Databricks to view.

Tranche de prix (USD),Taux moyen d'avis positifs (%),Nombre de jeux
< 5,75.4,22466
5 - 15,78.2,17213
15 - 30,77.7,5401
30 - 60,77.8,1034
> 60,77.6,113


Nous avons trois résultats convergents qui répondent à la question :
- le coefficient de corrélation, ou coefficient de Pearson, est quasi nul (0,0375),
- le taux moyen d'avis positifs quasi constant par tranche de prix (75,4 à 78,2%),
- une moyenne globale de 76,8%.

Ces trois éléments indiquent que le prix n'est pas un facteur discriminant de la satisfaction des joueurs.

#### 4.1.2 Les jeux gratuits génèrent-ils plus de volume d'avis que les jeux payants ?

In [0]:
popularity_df = df \
    .filter(F.col("nb_reviews").isNotNull()) \
    .select(
        F.col("nb_reviews"),
        F.col("price_USD"),
        F.col("positive").cast(DoubleType()).alias("positive"),
        F.col("negative").cast(DoubleType()).alias("negative")
    ) \
    .withColumn(
        "modèle",
        F.when(F.col("price_USD") == 0, "Gratuit").otherwise("Payant")
    )

popularity_agg_df = popularity_df \
    .groupBy("modèle") \
    .agg(
        F.round(100 * F.count("*") / NB_GAMES, 2).alias("Nombre de jeux (%)"),
        F.round(F.mean(F.col("nb_reviews")), 0).alias("Nombre moyen d'avis"),
        F.expr("percentile_approx(nb_reviews, 0.5)").alias("Nombre d'avis médian"),
        F.expr("percentile_approx(nb_reviews, 0.9)").alias("Nombre d'avis P90"),
        F.max(F.col("nb_reviews")).alias("Nombre d'avis maximum")
    ) \
    .orderBy("modèle")
 
display(popularity_agg_df)
display(
    popularity_df \
        .filter(F.col("modèle") == "Gratuit") \
        .select("nb_reviews", "modèle") \
        .withColumn(
            "log('Nombre d'avis' + 1)",
            F.round(F.log(F.col("nb_reviews") + 1), 2)
        ) \
        .withColumnRenamed("nb_reviews", "Nombre d'avis") \
        .select("log('Nombre d'avis' + 1)")
)
display(
    popularity_df \
        .filter(F.col("modèle") == "Payant") \
        .select("nb_reviews", "modèle") \
        .withColumn(
            "log('Nombre d'avis' + 1)",
            F.round(F.log(F.col("nb_reviews") + 1), 2)
        ) \
        .withColumnRenamed("nb_reviews", "Nombre d'avis") \
        .select("log('Nombre d'avis' + 1)")
)

Databricks visualization. Run in Databricks to view.

modèle,Nombre de jeux (%),Nombre moyen d'avis,Nombre d'avis médian,Nombre d'avis P90,Nombre d'avis maximum
Gratuit,13.97,3468.0,47,1569,6730438
Payant,86.03,1428.0,24,827,1442644


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

log('Nombre d'avis' + 1)
4.44
0.69
5.98
4.84
4.19
2.08
2.56
2.2
2.3
2.08


Databricks visualization. Run in Databricks to view.

log('Nombre d'avis' + 1)
12.24
3.5
8.45
7.43
0.69
7.3
3.22
1.95
3.81
1.95


Les jeux gratuits génèrent nettement plus de volume d'avis que les payants.

La moyenne des jeux gratuits est 2,4 fois plus grande que celle des payants. Cet écart pourrait provenir de quelques jeux à fort succès. Mais la médiane confirme la tendance : 47 avis pour le jeu gratuit contre 24 pour le payant, soit un rapport également proche de 2. La moyenne et la médiane vont dans le même sens : cela indique un écart structurel et non tiré par les extrêmes. Le P90 (1 569 vs 827) le confirme encore : même hors du top 10%, le gratuit est deux fois plus visible.

L'asymétrie est extrême dans les deux groupes : une médiane à 47 face à une moyenne à 3 468, pour le modèle gratuit, montre qu'une minorité de jeux concentre l'essentiel des avis. Le constat est identique pour le jeu payant (24 vs 1 428) : le modèle économique ne corrige pas la tendance.

Le gratuit ne rend pas un jeu populaire. En supprimant la barrière du prix, il élargit la base de joueurs susceptibles de laisser un avis. Le volume d'avis mesure une taille d'audience touchée, pas une qualité ni un succès commercial. Enfin, le modèle gratuit ne pèse que pour 14% du catalogue. Ce groupe est minoritaire mais aux effectifs largement suffisants pour conclure.

### 4.2 Analyse des développeurs

#### 4.2.1 Y a-t-il une différence de réception entre les jeux où l'éditeur est aussi le développeur (studios indépendants / auto-édition) et ceux portés par un éditeur tiers ?

In [0]:
dev_pub_df = df \
    .filter(
        (F.col("developer").isNotNull()) & 
        (F.col("publisher").isNotNull()) &
        (F.col("developer") != "") & 
        (F.col("publisher") != "")
    ) \
    .select(
        F.col("developer"),
        F.col("publisher"),
        F.col("positive_ratio_PCT"),
        F.col("nb_reviews")
    ) \
    .withColumn(
        "Modèle d'édition",
        F.when(F.lower(F.col("developer")) == F.lower(F.col("publisher")), "Auto-édition") \
        .otherwise("Editeur tiers")
    ) \
    .groupBy("Modèle d'édition") \
    .agg(
        F.count("*").alias("Nombre de jeux"),
        F.round(F.mean("positive_ratio_PCT"), 1).alias("Taux moyen d'avis positifs (%)"),
        F.round(F.expr("percentile_approx(positive_ratio_PCT, 0.5)"), 1).alias("Taux médian d'avis positifs (%)"),
        F.round(F.expr("percentile_approx(nb_reviews, 0.5)"), 0).alias("Nombre d'avis médian")
    )

display(dev_pub_df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Modèle d'édition,Nombre de jeux,Taux moyen d'avis positifs (%),Taux médian d'avis positifs (%),Nombre d'avis médian
Auto-édition,37490,73.0,79.6,20
Editeur tiers,17958,74.5,80.0,54


Sur le plan de la réception, la différence est négligeable. Les jeux portés par un éditeur tiers obtiennent 74,5% d'avis positifs en moyenne contre 73,0% pour les jeux auto-édités, et l'écart se réduit encore sur la médiane (80,0% contre 79,6%). Que l'on regarde la moyenne ou la médiane, les deux modèles convergent vers la même conclusion : passer par un éditeur tiers n'améliore pas la qualité perçue d'un jeu.

En revanche, un écart net apparaît sur la visibilité. Le jeu médian édité par un tiers cumule 54 avis, contre seulement 20 pour le jeu auto-édité médian, soit plus du double. Le volume d'avis étant une mesure de l'audience touchée, cela suggère qu'un éditeur tiers apporte surtout une force de distribution et de visibilité, pas un gain de qualité.

L'autre enseignement structurel est la répartition : l'auto-édition domine largement le catalogue Steam (68% des jeux), ce qui reflète l'accessibilité de la plateforme aux studios indépendants publiant sans intermédiaire.

**Remarque :** Une limite est à mentionner dans cette analyse. La comparaison repose sur une égalité quasi stricte (chaîne de caractères converties en lettres minuscules) entre les noms de développeur et d'éditeur. Ainsi, par exemple, les variantes de nommage "0 Deer Soft" vs "0 Deer Soft Partnership" classent par défaut en "éditeur tiers".

### 4.3 Internationalisation

#### 4.3.1 Le nombre de langues supportées est-il lié au volume d'avis ou aux notes ? Autrement dit, un effort de localisation s'accompagne-t-il d'une meilleure visibilité ?

In [0]:
languages_df = df \
    .withColumn("nb_languages", F.size(F.col("languages"))) \
    .filter(
        (F.col("nb_languages") > 0) &
        (F.col("nb_reviews").isNotNull()) &
        (F.col("positive_ratio_PCT").isNotNull())
    ) \
    .select(
        F.col("nb_languages"),
        F.col("nb_reviews"),
        F.col("positive_ratio_PCT")
    )

correlations = languages_df \
    .agg(
        F.corr("nb_languages", "nb_reviews").alias("corr_reviews"),
        F.corr("nb_languages", "positive_ratio_PCT").alias("corr_ratio")
    ) \
    .collect()[0]

corr_nb_lang_vs_nb_reviews = round(correlations["corr_reviews"], 4)
corr_nb_lang_vs_positive_ratio = round(correlations["corr_ratio"], 4)

corr_prefix = "Corrélation entre nombre de langues et "
print(f"{corr_prefix}nombre d'avis :        {corr_nb_lang_vs_nb_reviews}")
print(f"{corr_prefix}taux d'avis positifs : {corr_nb_lang_vs_positive_ratio}")

languages_df = languages_df \
.withColumn(
    "language_range_cat",
    F.when(F.col("nb_languages") == 1, 1)
     .when(F.col("nb_languages") <= 3, 2)
     .when(F.col("nb_languages") <= 7, 3)
     .when(F.col("nb_languages") <= 15, 4)
     .otherwise(5)
) \
.groupBy("language_range_cat") \
.agg(
    F.count("*").alias("Nombre de jeux"),
    F.round(F.expr("percentile_approx(nb_reviews, 0.5)"), 0).alias("Nombre d'avis médian"),
    F.round(F.expr("percentile_approx(positive_ratio_PCT, 0.5)"), 1).alias("Taux médian d'avis positifs (%)")
) \
.orderBy("language_range_cat") \
.withColumn(
    "Nombre de langues",
    F.when(F.col("language_range_cat") == 1, "1")
     .when(F.col("language_range_cat") == 2, "2-3")
     .when(F.col("language_range_cat") == 3, "4-7")
     .when(F.col("language_range_cat") == 4, "8-15")
     .otherwise("16+")
) \
.drop("language_range_cat") \
.select(
    "Nombre de langues",
    "Nombre de jeux",
    "Nombre d'avis médian",
    "Taux médian d'avis positifs (%)"
)

display(languages_df)

Corrélation entre nombre de langues et nombre d'avis :        0.0854
Corrélation entre nombre de langues et taux d'avis positifs : 0.0505


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Nombre de langues,Nombre de jeux,Nombre d'avis médian,Taux médian d'avis positifs (%)
1,29665,16,78.6
2-3,10966,23,78.9
4-7,6981,93,80.0
8-15,6307,237,81.4
16+,1771,39,81.8


Le nombre de langues est lié au volume d'avis, mais de façon non linéaire. Le jeu médian monolingue cumule 16 avis. Ce chiffre grimpe à 93 (4-7 langues), puis 237 (8-15 langues), soit près de 15 fois plus. Traduire un jeu dans davantage de langues s'accompagne donc d'une audience nettement plus large jusqu'à un point : la tranche "16 langues et +" casse la tendance et rechute à 39 avis. Au-delà d'un certain seuil, sur-localiser ne garantit plus la visibilité.

Sur les notes, l'effet est réel mais modeste : le taux médian d'avis positifs progresse régulièrement de 78,6% à 81,8%, soit environ 3% sur tout le spectre.

Le point crucial est que les corrélations de Pearson sont quasi nulles (0,085 pour le volume, 0,051 pour le taux). Néanmoins cela ne contredit pas le tableau car ce coefficient mesure un lien linéaire individuel, écrasé ici par l'asymétrie des volumes. Au niveau d'un jeu donné, le nombre de langues ne prédit rien mais les jeux très traduits sont typiquement plus visibles.

Pour finir, la moitié du catalogue n'existe qu'en une langue, indiquant une production plutôt indépendante. La causalité reste indéterminable en EDA : on traduit souvent un jeu en multiples langues parce qu'on anticipe son succès.

### 4.4 Saisonnalité des sorties

#### 4.4.1 Y a-t-il des mois ou des saisons privilégiés pour les sorties de jeux ? Les sorties de fin d'année (période de Noël) sont-elles plus nombreuses ou mieux notées ?

In [0]:
dates_df = df \
    .filter(F.col("release_date").isNotNull()) \
    .withColumn("month", F.month("release_date")) \
    .withColumn("day", F.day("release_date")) \
    .withColumn("month_alpha", F.expr("date_format(release_date, 'MMM')")) \
    .withColumn("dayofyear_alpha", F.expr("date_format(release_date, 'MM-dd')")) \
    .select(
        F.col("month"),
        F.col("month_alpha"),
        F.col("day"),
        F.col("positive_ratio_PCT"),
        F.col("dayofyear_alpha")
    )

releases_per_month_df = dates_df \
    .filter(F.col("positive_ratio_PCT").isNotNull()) \
    .groupBy("month_alpha", "month") \
    .agg(
        F.count("*").alias("Nombre de sorties"),
        F.round(F.expr("percentile_approx(positive_ratio_PCT, 0.5)"), 1).alias("Taux médian d'avis positifs (%)")
    ) \
    .orderBy("month") \
    .drop("month") \
    .withColumnRenamed("month_alpha", "Mois")

print(
    "Moyenne du nombre total de sorties par mois : " +
    f"{int(releases_per_month_df.select(F.mean('Nombre de sorties').alias('moy')).first()['moy'] + 0.5)}"
)

display(releases_per_month_df)

Moyenne du nombre total de sorties par mois : 4633


Databricks visualization. Run in Databricks to view.

Mois,Nombre de sorties,Taux médian d'avis positifs (%)
Jan,3877,78.6
Feb,4219,79.0
Mar,4708,79.2
Apr,4470,79.7
May,4731,79.6
Jun,4212,79.7
Jul,4767,78.2
Aug,5077,79.6
Sep,5177,80.0
Oct,5549,81.2


**Remarques préliminaires :**
- L'axe de droite du graphique ci-dessus a été volontairement borné entre 76% et 84% (et ne part donc pas de zéro) afin de rendre visibles des écarts faibles. Cela doit être signalé pour ne pas exagérer optiquement la variation des notes.
- Le nombre de sorties affiché pour chaque mois n'est pas une moyenne mensuelle mais un nombre total de sorties pour chaque mois (cumul du même mois sur l'ensemble des années).

Il existe un effet saisonnier sur le **volume** de sorties, mais il ne correspond pas à l'idée reçue. Sur environ 55 600 jeux datés, le pic se situe à l'**automne, en octobre** (5 549 sorties), après une montée progressive des sorties depuis le mois de juin. À l'inverse, le creux est en **janvier** (3 877). L'écart entre le mois le plus chargé et le plus calme atteint ~43%, ce qui est significatif.

Le résultat contre-intuitif concerne **décembre** : avec 4 248 sorties, le mois de Noël est *en dessous* de la moyenne mensuelle (4 633), loin d'être un pic. Les éditeurs sortent visiblement leurs jeux *avant* la période d'achats de fin d'année, en septembre-octobre, plutôt que pendant. Sortir en pleine concurrence de décembre est au contraire évité.

Sur les **notes**, l'effet est négligeable. Le taux médian d'avis positifs reste compris entre 78,2% (juillet) et 81,2% (octobre), soit 3% d'amplitude. Octobre cumule donc le plus de sorties *et* la meilleure note médiane, mais l'écart de qualité est trop faible pour conclure : le mois de sortie n'influence pas la réception. Les sorties de Noël ne sont donc ni plus nombreuses, ni mieux notées.

#### 4.4.2 Y a-t-il des dates de sortie "maudites" (ex. : 1er avril, 31 octobre et 25 décembre) ?

In [0]:
dayofyear_df = dates_df \
    .groupBy("dayofyear_alpha") \
    .agg(
        F.count("*").alias("Nombre de sorties"),
        #F.round(F.expr("percentile_approx(positive_ratio_PCT, 0.5)"), 1).alias("Taux médian d'avis positifs (%)")
    ) \
    .orderBy("dayofyear_alpha") \
    .withColumnRenamed("dayofyear_alpha", "Jour de l'année (format MM-JJ)")

col = "Nombre de sorties"
dayofyear_df \
    .select(col) \
    .summary() \
    .select(
        "summary",
        *[
            F.when(
                F.col("summary").isin(["mean", "stddev"]),
                F.lpad(F.round(F.col(c).cast(DoubleType()), 1).cast("string"), len(col), " ")
            ) \
            .otherwise(
                F.lpad(F.col(c), len(col), " ")
            ).alias(c)
            for c in [col]
        ]
    ) \
    .show(truncate=False)

display(dayofyear_df)

+-------+-----------------+
|summary|Nombre de sorties|
+-------+-----------------+
|count  |              366|
|mean   |            151.9|
|stddev |             32.6|
|min    |               25|
|25%    |              132|
|50%    |              152|
|75%    |              171|
|max    |              267|
+-------+-----------------+



Databricks visualization. Run in Databricks to view.

Jour de l'année (format MM-JJ),Nombre de sorties
01-01,81
01-02,69
01-03,90
01-04,95
01-05,123
01-06,125
01-07,126
01-08,99
01-09,86
01-10,106


L'idée d'une date "maudite" au sens superstitieux ne résiste pas aux données. En revanche, une malédiction d'un autre type, purement stratégique, apparaît très nettement. Sur une moyenne de 152 sorties par jour, l'examen des trois dates symboliques candidates donne un résultat sans ambiguïté.

Le **1er avril** n'a aucun effet : 174 sorties, soit légèrement *au-dessus* de la moyenne. Malgré son potentiel thématique (jeux parodiques, blagues), cette date n'est ni évitée ni privilégiée. Le **31 octobre** (Halloween) est même un jour plutôt actif, avec 202 sorties, au-delà de la moyenne. Là encore, aucune malédiction : sortir un jeu, fût-il d'horreur, le jour d'Halloween n'est pas une pratique remarquable dans les données.

Le **25 décembre**, en revanche, est l'un des jours les plus vides de l'année : 56 sorties seulement, soit un tiers de la normale. Mais ce n'est pas Noël en tant que date symbolique qui est en cause : c'est toute la **période du 23 au 31 décembre** qui s'effondre (49 sorties le 26, 59 le 30). Le creux est un plateau d'une semaine, pas un jour isolé. C'est la signature d'une cause structurelle, non d'une superstition.

L'explication est double et parfaitement rationnelle. D'une part, les **fêtes de fin d'année** : joueurs en famille, attention médiatique nulle, aucun intérêt à lancer un titre. D'autre part, et surtout, les **soldes d'hiver de Steam**, qui concentrent l'attention et le budget des joueurs vers les promotions du catalogue existant. Sortir une nouveauté dans cette fenêtre revient à la condamner à l'invisibilité.

La conclusion qui clôt cette question est donc à prendre avec le sourire : il n'existe pas de date maudite par superstition, mais une **malédiction logistique** bien réelle : on ne sort pas un jeu quand les joueurs sont en vacances ou happés par les soldes.

---

## Conclusion et recommandations à Ubisoft

### Conclusion

Steam est un marché massif, mature et fortement concurrentiel. Sa croissance, explosive entre 2013 et 2018, s'est nettement ralentie depuis 2019, indépendamment du Covid : le catalogue arrive à saturation et la rareté de l'attention devient le vrai facteur limitant.

L'analyse fait ressortir trois constats structurants. Premièrement, le prix n'est pas un levier : il n'a aucun lien avec la satisfaction (corrélation de 0,0375, taux d'avis positifs stable autour de 77% quelle que soit la tranche de prix). Le marché est déjà dominé par les bas prix (médiane proche de 5 USD) et se battre sur ce terrain n'apporte rien. Deuxièmement, la visibilité, mesurée par le volume d'avis, ne dépend ni du prix ni de la qualité intrinsèque, mais de l'audience atteinte : un éditeur tiers multiplie par 2,7 le volume médian d'avis (54 contre 20 en auto-édition) sans améliorer la note. Troisièmement, le genre Action concentre le plus gros chiffre d'affaires total (58,8 Md USD), tout en restant correctement noté et au cœur du savoir-faire d'Ubisoft.

Enfin, des leviers opérationnels à fort effet et à coût faible sont identifiés : une localisation calibrée multiplie par 15 l'audience médiane, et la fenêtre de sortie crée jusqu'à 43% d'écart de volume. La popularité d'un jeu se construit donc moins par son prix que par sa qualité, sa visibilité et son timing.

### Recommandations

**1. Capitaliser sur le genre Action, viser le RPG/MMO pour la rentabilité unitaire.**
L'Action est le genre le plus rémunérateur en volume (58,8 Md USD de CA total, valeur estimée de type "ordre de grandeur"), correctement noté et déjà au cœur du catalogue Ubisoft. Pour maximiser le revenu par titre, le RPG (2,85 M USD/jeu, estimé) et le Massively Multiplayer (4,06 M USD/jeu) offrent un potentiel unitaire supérieur, au prix d'une exigence de réception plus forte (le MMO étant le genre le plus mal noté). Arbitrage recommandé : un titre Action à forte composante RPG.

**2. Différencier par la qualité perçue, jamais par le prix.**
Le prix n'a aucun effet sur la satisfaction ni sur le succès. Tout euro investi dans une baisse de prix est perdu ; il doit être réalloué à la qualité (finition, contenu, polish), seul facteur réellement corrélé à une bonne réception et à la recommandation. Positionner le jeu sur sa valeur, pas sur son tarif.

**3. Calibrer la localisation entre 8 et 15 langues.**
L'anglais est impératif. Passer du monolingue à 8-15 langues multiplie par 15 l'audience médiane (16 à 237 avis). Au-delà de 16 langues, le rendement s'effondre : sur-localiser est un gaspillage. Cibler en priorité allemand, français, russe, chinois simplifié, espagnol et japonais, qui couvrent les marchés les plus représentés.

**4. Verrouiller une fenêtre de sortie en septembre-octobre.**
Octobre est le pic d'activité (5 549 sorties) et la meilleure note médiane. Viser une sortie septembre-octobre, avant la concurrence et les achats de fin d'année. Proscrire la période du 23 au 31 décembre (soldes d'hiver Steam, invisibilité quasi totale) ainsi que le creux de janvier. Aucune date "maudite" superstitieuse n'existe : la contrainte est logistique, pas symbolique.

**5. Concentrer le développement sur Windows, ajouter Mac, et activer la force de distribution d'Ubisoft.**
Windows est incontournable (99,97% du catalogue) ; Mac est le seul portage secondaire au rendement justifiable (22,93%), Linux restant marginal (15,19%) et réservé à des cas ciblés. Surtout, la visibilité étant le facteur décisif et indépendant de la qualité, Ubisoft doit pleinement exploiter sa puissance d'éditeur établi (marketing, référencement, communauté) : c'est l'avantage structurel le plus différenciant face aux 68% de jeux auto-édités du marché.

# Annexe - Cellules pour dashboard

Inutile de consulter cette partie. SVP consultez directement le dashboard.

## 📊 Contexte & objectifs

### Positionnement

Travail pour le compte d'Ubisoft qui souhaite sortir un nouveau jeu vidéo.

### Objectifs

- Comprendre les facteurs qui influencent la popularité ou les ventes.
- Dresser une analyse globale du marché du jeu vidéo.
- Adopter plusieurs niveaux d'analyse (macroscopique, par genres, par plateformes).

## 📂 Dataset

Le [dataset](https://full-stack-bigdata-datasets.s3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json) recense les jeux disponibles sur la marketplace de Steam, service de distribution numérique et boutique de jeux vidéo.

Chaque enregistrement décrit un jeu et ses caractéristiques : éditeur, prix, genres, plateformes, notes des utilisateurs, etc.

Il s'agit d'un fichier semi-structuré au format JSON, doté d'un schéma imbriqué (structures et tableaux sur plusieurs niveaux).

## 🔍 Scope

Analyse exploratoire (EDA) réalisée avec Databricks et PySpark.

\\[
    \text{CA} \quad = \quad \text{prix de vente} \quad \times \quad \text{nombre de possesseurs}
\\]

## 🎯 Conclusion & recommandations à Ubisoft

### Trois constats structurants

- **Le prix n'est pas un levier.**
    - Corrélation prix / satisfaction ≈ 0
    - le taux d'avis positifs reste stable autour de **77%** sur toutes les tranches de prix.
- **La visibilité prime sur la qualité.**
    - Un éditeur tiers multiplie par **2,7** le volume médian d'avis (54 vs 20) sans améliorer la note.
- **L'Action domine le marché.**
    - **58,8 Md USD** de CA total
    - Genre correctement noté
    - Déjà au cœur du savoir-faire Ubisoft.

### Cinq recommandations

| # | Recommandation | Chiffre-clé |
|---|---|---|
| 1 | **Capitaliser sur l'Action, intégrer une forte composante RPG** pour la rentabilité unitaire | RPG : **2,85 M USD / jeu** |
| 2 | **Différencier par la qualité perçue, jamais par le prix** : tout euro de baisse de prix est perdu | Médiane marché ≈ **5 USD** |
| 3 | **Calibrer la localisation entre 8 et 15 langues** (EN, DE, FR, RU, ZH, ES, JA en priorité) | Audience médiane × **15** |
| 4 | **Sortir en septembre-octobre**, éviter le 23-31 décembre et janvier | Octobre : **5 549** sorties, pic d'activité |
| 5 | **Concentrer sur Windows, ajouter Mac, exploiter la force de distribution Ubisoft** | Windows : **99,97%** du catalogue |

> **Idée centrale.** Sur un marché saturé où 68% des jeux sont auto-édités, la popularité d'un jeu se joue moins par son prix que par sa **qualité**, sa **visibilité** et son **timing** — trois leviers sur lesquels Ubisoft possède un avantage structurel.